# GSE295514 - RDS を読む（R kernel）

`GSE295514_ALS_mouse_brain.rds` を対話的に開き、class / assays / `meta.data` を確認してから、Python が読める中間ファイルを `data/intermediate_from_r/GSE295514/` に書き出す:

- `counts.mtx`（genes x cells, Matrix Market）
- `metadata.csv`
- `genes.csv`（gene_symbol）
- `barcodes.csv`（cell_id）

Python ノートブック 01 が `io_dense.read_from_r_intermediate(...)` でこれを読む。

必要: **R Jupyter kernel (IRkernel)** と `Matrix`、対象に応じて `Seurat`/`SeuratObject` または `SingleCellExperiment`。

In [ ]:
library(Matrix)

# rds を読み込み、まずクラスを確認
rds_path <- "../../data/raw/GSE295514/GSE295514_ALS_mouse_brain.rds"
obj <- readRDS(rds_path)

class(obj)

In [ ]:
names(obj)
tryCatch(slotNames(obj), error = function(e) e$message)

## Seurat の場合

In [ ]:
# library(Seurat)
# Assays(obj)            # assay 一覧
# DefaultAssay(obj)
# head(obj@meta.data)    # meta.data の列
# colnames(obj@meta.data)

In [ ]:
# counts <- GetAssayData(obj, assay = DefaultAssay(obj), slot = "counts")
# dim(counts); counts[1:5, 1:5]

## SingleCellExperiment の場合

In [ ]:
# library(SingleCellExperiment)
# assayNames(obj)
# counts <- assay(obj, "counts")
# colData(obj); rowData(obj)

## Python 用の中間ファイルを書き出す

上のどちらかで `counts`(genes x cells) と `meta`(細胞メタ) を作ってから実行。

In [ ]:
out_dir <- "../../data/intermediate_from_r/GSE295514"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# --- オブジェクトに応じて counts / meta を取り出す ---
if (inherits(obj, "Seurat")) {
    counts <- GetAssayData(obj, assay = DefaultAssay(obj), slot = "counts")
    meta <- obj@meta.data
} else if (inherits(obj, "SingleCellExperiment")) {
    counts <- SummarizedExperiment::assay(obj, "counts")
    meta <- as.data.frame(SummarizedExperiment::colData(obj))
} else {
    stop(paste("未対応のクラス:", paste(class(obj), collapse = ", ")))
}

counts <- as(counts, "CsparseMatrix")
writeMM(counts, file.path(out_dir, "counts.mtx"))
write.csv(meta, file.path(out_dir, "metadata.csv"))
write.csv(data.frame(gene_symbol = rownames(counts)),
          file.path(out_dir, "genes.csv"), row.names = FALSE)
write.csv(data.frame(cell_id = colnames(counts)),
          file.path(out_dir, "barcodes.csv"), row.names = FALSE)

cat("中間ファイルを書き出しました:", out_dir, "\n")
dim(counts)

この後 **notebooks/python/01** の GSE295514 セル （`io_dense.read_from_r_intermediate(...)`）で読み込む。